In [4]:
import json
import glob
import os
from pathlib import Path
import time
import os
from dotenv import load_dotenv

load_dotenv()

def consolidar_jsons(fonte, cidade, PASTA_DADOS):
    
    now = time.strftime("%Y-%m")
    
    padrao_busca = str(PASTA_DADOS / f'{cidade}_{fonte}_*.json')
    
    arquivos_json = glob.glob(padrao_busca)
    
    if not arquivos_json:
        print("Nenhum arquivo encontrado com o padrão especificado.")
        return

    dados_consolidados = []
    total_arquivos = len(arquivos_json)

    print(f"Iniciando a união de {total_arquivos} arquivos...")

    for i, caminho in enumerate(arquivos_json, 1):
        nome_base = os.path.basename(caminho)
        with open(caminho, 'r', encoding='utf-8') as f:
            try:
                conteudo = json.load(f)
                # Verifica se o conteúdo é uma lista (padrão do seu scraper)
                if isinstance(conteudo, list):
                    dados_consolidados.extend(conteudo)
                else:
                    dados_consolidados.append(conteudo)
                
                print(f"[{i}/{total_arquivos}] Adicionado: {nome_base} ({len(conteudo)} itens)")
            except Exception as e:
                print(f"Erro ao ler {nome_base}: {e}")

    # 2. Salva o arquivo final consolidado
    nome_final = f'{cidade}_{fonte}_{now}.json'
    
    caminho_final = PASTA_DADOS / nome_final

    try:
        with open(caminho_final, 'w', encoding='utf-8') as f_out:
            json.dump(dados_consolidados, f_out, indent=4, ensure_ascii=False)
        
        # --- VALIDAÇÃO DE SEGURANÇA ---
        tamanho_final = os.path.getsize(caminho_final)
        
        """if tamanho_final > 0 and len(dados_consolidados) > 0:
            print(f"✅ Consolidação concluída: {len(dados_consolidados)} registros.")
            print(f"📦 Arquivo gerado: {nome_final} ({tamanho_final / 1024 / 1024:.2f} MB)")
            
            # 4. Deleta os arquivos anteriores apenas se o final estiver OK
           
            print("🗑️ Removendo arquivos temporários (fatias)...")
            for arquivo_velho in arquivos_json:
                try:
                    os.remove(arquivo_velho)
                    print(f"   Excluído: {os.path.basename(arquivo_velho)}")
                except Exception as e:
                    print(f"   Erro ao excluir {arquivo_velho}: {e}")
        
        
            
            print("✨ Limpeza concluída com sucesso!")
        else:
            print("⚠️ Erro crítico: O arquivo final parece estar vazio. Abortando exclusão.")"""

    except Exception as e:
        print(f"❌ Erro ao salvar arquivo consolidado: {e}")


In [5]:

import asyncio
import sys
from pathlib import Path
import time

BASE_DIR = Path.cwd().parent

PASTA_DADOS = BASE_DIR / 'dados' / 'balneario_camboriu'

#consolidar_jsons('olx', 'balneario_camboriu', PASTA_DADOS)


In [6]:
PASTA_DADOS

WindowsPath('c:/Users/jefer/Documents/Ciencia-de-dados/Preco-Imoveis/dados/balneario_camboriu')

In [7]:
import pandas as pd
import warnings
import logging
import asyncio
from funcoes_limpando_dados_imoveis import (limpar_valor_iptu,
                                            limpar_banheiros, 
                                            limpar_metragem, 
                                            limpar_vagas,  
                                            #limpa_endereco_apply, 
                                            limpar_valor_condominio, 
                                            converter_para_data, 
                                            classificar_tipo_imovel, 
                                            reclassificar_outros, 
                                            preencher_todas_coordenadas,
                                            main_example, 
                                            limpar_valor_venda, 
                                            limpar_quartos, 
                                            pirabeiraba_dona_francisca, 
                                            geocodificar_dataframe,
                                            limpa_endereco_apply_zap, 
                                            limpa_endereco_apply_chave_mao, 
                                            limpa_endereco_apply_olx,)
import time
from datetime import datetime
import json
from pathlib import Path

import asyncio
import sys
from pathlib import Path
import time

cidade = 'joinville'

estado = 'sc'

BASE_DIR = Path.cwd().parent

PASTA_DADOS = BASE_DIR / 'dados' / cidade

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

logger = logging.getLogger(__name__)

warnings.filterwarnings("ignore")

start_time = time.time()

async def limpando_dados_cidades(pd_data, batch, cidade_limpeza = 'joinville', estado_limpeza = 'sc', cidade_localizacao = 'Joinville', estado_localizacao = 'SC',  tipo_async=True,  pais='Brasil'): 
       
    logger.info("Iniciando o processo de limpeza de dados de imóveis...")    
    
    pd_data = pd_data.drop_duplicates(subset=['url'])

    pd_data = pd_data[pd_data['valor_imovel'].notna()]

    pd_data_sem_nulos = pd_data.dropna(thresh=10)

    logger.info(f"Removendo linhas com muitos valores faltantes. Registros restantes: {pd_data_sem_nulos.shape}")

    #endereco_dividido = pd_data_sem_nulos['endereco'].apply(limpa_endereco_apply)

    #pd_data_endereco_dividido = pd.concat([pd_data_sem_nulos, endereco_dividido], axis=1).drop('endereco', axis=1)

    #pd_data_endereco_dividido['bairro'] = pd_data_endereco_dividido['bairro'].apply(pirabeiraba_dona_francisca)

    #logger.info("Coluna 'endereco' dividida em 'rua', 'bairro', 'cidade' e 'estado'...")
    
    pd_data_estado = pd_data_sem_nulos.copy()

    pd_data_estado = pd_data_estado[pd_data_estado['estado'] == estado_limpeza]

    logger.info(f"Removendo linhas com estado diferente de {estado_limpeza}. Registros restantes: {pd_data_estado.shape}")

    pd_data_metragem = pd_data_estado.copy()

    pd_data_metragem['metragem'] = pd_data_metragem['metragem'].apply(limpar_metragem)

    logger.info(f"Coluna 'metragem' limpa. Registros restantes: {pd_data_metragem.shape}")

    pd_data_valor_imovel = pd_data_metragem.copy()

    try:
        pd_data_valor_imovel['valor_venda'] = pd_data_valor_imovel['valor_venda'].apply(limpar_valor_venda)
    except:
        pd_data_valor_imovel['valor_imovel'] = pd_data_valor_imovel['valor_imovel'].apply(limpar_valor_venda)


    logger.info(f"Coluna 'valor_venda' limpa. Registros restantes: {pd_data_valor_imovel.shape}")

    pd_data_valor_condominio = pd_data_valor_imovel.copy()

    pd_data_valor_condominio['condominio'] = pd_data_valor_condominio['condominio'].apply(limpar_valor_condominio)

    logger.info(f"Coluna 'condominio' limpa. Registros restantes: {pd_data_valor_condominio.shape}")

    pd_data_valor_iptu = pd_data_valor_condominio.copy()

    pd_data_valor_iptu['iptu'] = pd_data_valor_iptu['iptu'].apply(limpar_valor_iptu)
    

    logger.info(f"Coluna 'iptu' limpa. Registros restantes: {pd_data_valor_iptu.shape}")

    pd_data_ano_publicacao = pd_data_valor_iptu.copy()

    pd_data_ano_publicacao['data_criacao'] = pd_data_ano_publicacao['data_criacao'].apply(converter_para_data)

    pd_data_ano_publicacao['dias_publicacao'] = (pd.to_datetime(datetime.now().strftime('%Y-%m-%d')) - pd.to_datetime(pd_data_ano_publicacao['data_criacao'], format='%d/%m/%Y')).dt.days
    
    logger.info(f"Coluna 'data_criacao' limpa. Registros restantes: {pd_data_ano_publicacao.shape}")

    pd_data_banheiros = pd_data_ano_publicacao.copy()
    
    pd_data_banheiros['banheiros'] = pd_data_banheiros['banheiros'].apply(limpar_banheiros)

    logger.info(f"Coluna 'banheiros' limpa. Registros restantes: {pd_data_banheiros.shape}")

    pd_data_quartos = pd_data_banheiros.copy()

    pd_data_quartos['quartos'] = pd_data_quartos['quartos'].apply(limpar_quartos)

    logger.info(f"Coluna 'quartos' limpa. Registros restantes: {pd_data_quartos.shape}")

    pd_data_garagem = pd_data_quartos.copy()

    #pd_data_garagem['vagas'] = pd_data_garagem['vagas'].replace('--', 0).astype('int64')

    pd_data_garagem['vagas'] = pd_data_garagem['vagas'].apply(limpar_vagas)

    logger.info(f"Coluna 'vagas' limpa. Registros restantes: {pd_data_garagem.shape}")

    pd_data_tipo_imovel = pd_data_garagem.copy()

    pd_data_tipo_imovel['tipo_imovel'] = pd_data_tipo_imovel['titulo'].apply(classificar_tipo_imovel)

    mask = pd_data_tipo_imovel['tipo_imovel'] == 'outros'

    pd_data_tipo_imovel.loc[mask, 'tipo_imovel'] = (
        pd_data_tipo_imovel.loc[mask, 'descricao']
        .apply(reclassificar_outros)
    )

    logger.info(f"Coluna 'tipo_imovel' classificada. Registros restantes: {pd_data_tipo_imovel.shape}")

    pd_data_long_lat = pd_data_tipo_imovel.copy()
    
    if tipo_async:
        pd_data_lat_log_completo = await preencher_todas_coordenadas(pd_data_long_lat, batch_size=batch, cidade=cidade_localizacao, estado=estado_localizacao, pais=pais)
    else:
        pd_data_lat_log_completo = geocodificar_dataframe(pd_data_long_lat, cidade=cidade_localizacao, estado=estado_localizacao, pais=pais)

    logger.info(f"Todas as coordenadas preenchidas. Registros restantes: {pd_data_lat_log_completo.shape}")

    pd_data_lat_log_completo['preco_por_m2'] = pd_data_lat_log_completo['valor_imovel'] / pd_data_lat_log_completo['metragem']

    logger.info(f'Coluna "preco_por_m2" criada. Registros restantes: {pd_data_lat_log_completo.shape}')

    def classificar_dentro_bairro(grupo):
        p25 = grupo["preco_por_m2"].quantile(0.25)
        p50 = grupo["preco_por_m2"].quantile(0.50)
        p75 = grupo["preco_por_m2"].quantile(0.75)
        
        def faixa(val):
            if val <= p25:
                return "barato"
            elif val <= p50:
                return "medio_baixo"
            if val <= p75:
                return "medio_alto"
            else:
                return "alto_padrao"

        grupo = grupo.copy()
        grupo["faixa"]       = grupo["preco_por_m2"].apply(faixa)
        grupo["p25_bairro"]  = p25
        grupo["p50_bairro"]  = p50
        grupo["p75_bairro"]  = p75
        return grupo

    pd_data_range_bairro_tipo_imovel = pd_data_lat_log_completo.groupby(["bairro", "tipo_imovel"], group_keys=False, ).apply(classificar_dentro_bairro)

    pd_data_range_bairro_tipo_imovel = pd.concat([pd_data_lat_log_completo, pd_data_range_bairro_tipo_imovel[['faixa', 'p25_bairro', 'p50_bairro', 'p75_bairro']]], axis=1)

    logger.info(f"Criando Faixas de preço por bairro e tipo de imóvel classificadas. Registros restantes: {pd_data_range_bairro_tipo_imovel.shape}")

    pd_data_range_bairro_tipo_imovel["desvio_mediana"] = round((pd_data_range_bairro_tipo_imovel["preco_por_m2"] - pd_data_range_bairro_tipo_imovel["p50_bairro"]) / pd_data_range_bairro_tipo_imovel["p50_bairro"],2)

    logger.info(f"Coluna 'desvio_mediana' criada. Registros restantes: {pd_data_range_bairro_tipo_imovel.shape}")
    
    return pd_data_range_bairro_tipo_imovel

In [8]:
def carregar_json(pasta_dados: Path, glob_pattern: str) -> tuple[pd.DataFrame, Path | None]:
    """
    Busca o arquivo mais recente pelo padrão e retorna um DataFrame.
    Retorna DataFrame vazio se não encontrar nenhum arquivo.
    """
    arquivos = list(pasta_dados.glob(glob_pattern))

    if not arquivos:
        logger.warning(f"Nenhum arquivo encontrado para o padrão: {glob_pattern}")
        return pd.DataFrame(), None

    arquivo = max(arquivos, key=lambda f: f.stem.split('_')[-1])
    
    logger.info(f"Arquivo encontrado: {arquivo.name}")

    try:
        with open(arquivo, 'r', encoding='utf-8') as f:
            data = json.load(f)
        return pd.DataFrame(data), arquivo

    except Exception as e:
        logger.error(f"Erro ao carregar {arquivo.name}: {e}")
        return pd.DataFrame(), arquivo

def deletar_arquivo(arquivo: Path | None):
    if arquivo and arquivo.exists():
        arquivo.unlink()
        logger.info(f"Arquivo deletado: {arquivo.name}")
        
def carregar_parquet(pasta_dados: Path, glob_pattern: str) -> tuple[pd.DataFrame, Path | None]:
    """
    Busca o arquivo .parquet mais recente pelo padrão e retorna um DataFrame.
    Retorna DataFrame vazio se não encontrar nenhum arquivo.
    """
    # Se o padrão ainda estiver como .json, trocamos para .parquet
    if '.json' in glob_pattern:
        glob_pattern = glob_pattern.replace('.json', '.parquet')

    arquivos = list(pasta_dados.glob(glob_pattern))

    if not arquivos:
        logger.warning(f"Nenhum arquivo encontrado para o padrão: {glob_pattern}")
        return pd.DataFrame(), None

    # Mantém sua lógica de pegar o arquivo com a data mais recente no nome
    try:
        arquivo = max(arquivos, key=lambda f: f.stem.split('_')[-1])
    except Exception:
        # Fallback caso o nome do arquivo não siga o padrão de data
        arquivo = max(arquivos, key=lambda f: f.stat().st_mtime)
    
    logger.info(f"Arquivo Parquet encontrado: {arquivo.name}")

    try:
        # O pandas lê o Parquet diretamente do caminho (Path ou str)
        # Não é necessário usar 'with open' pois o formato é binário
        df = pd.read_parquet(arquivo)
        return df, arquivo

    except Exception as e:
        logger.error(f"Erro ao carregar {arquivo.name}: {e}")
        return pd.DataFrame(), arquivo


def deletar_arquivo(arquivo: Path | None):
    # Esta função permanece igual, pois Path.unlink() deleta qualquer tipo de arquivo
    if arquivo and arquivo.exists():
        arquivo.unlink()
        logger.info(f"Arquivo deletado: {arquivo.name}")

In [9]:
import json
import glob
import os
from pathlib import Path
import time
import os
from dotenv import load_dotenv
import pandas as pd

load_dotenv()

def consolidar_jsons(fonte, cidade, PASTA_DADOS, bairro = None):
    
    now = time.strftime("%Y-%m")
    
    if bairro is not None:
        padrao_busca = str(PASTA_DADOS / f'{cidade}_{bairro}_{fonte}_*.json')
    
    else:
    
        padrao_busca = str(PASTA_DADOS / f'{cidade}_{fonte}_*.json')
    
    arquivos_json = glob.glob(padrao_busca)
    
    if not arquivos_json:
        print("Nenhum arquivo encontrado com o padrão especificado.")
        return

    dados_consolidados = []
    total_arquivos = len(arquivos_json)

    print(f"Iniciando a união de {total_arquivos} arquivos...")

    for i, caminho in enumerate(arquivos_json, 1):
        nome_base = os.path.basename(caminho)
        with open(caminho, 'r', encoding='utf-8') as f:
            try:
                conteudo = json.load(f)
                # Verifica se o conteúdo é uma lista (padrão do seu scraper)
                if isinstance(conteudo, list):
                    dados_consolidados.extend(conteudo)
                else:
                    dados_consolidados.append(conteudo)
                
                print(f"[{i}/{total_arquivos}] Adicionado: {nome_base} ({len(conteudo)} itens)")
            except Exception as e:
                print(f"Erro ao ler {nome_base}: {e}")

    if bairro is not None:
        nome_final = f'{cidade}_{bairro}_{fonte}_{now}.json'
    else:
        nome_final = f'{cidade}_{fonte}_{now}.json'
    
    caminho_final = PASTA_DADOS / nome_final

    try:
        with open(caminho_final, 'w', encoding='utf-8') as f_out:
            json.dump(dados_consolidados, f_out, indent=4, ensure_ascii=False)
        
        # --- VALIDAÇÃO DE SEGURANÇA ---
        tamanho_final = os.path.getsize(caminho_final)
        
        if tamanho_final > 0 and len(dados_consolidados) > 0:
            print(f"✅ Consolidação concluída: {len(dados_consolidados)} registros.")
            print(f"📦 Arquivo gerado: {nome_final} ({tamanho_final / 1024 / 1024:.2f} MB)")
            
            # 4. Deleta os arquivos anteriores apenas se o final estiver OK
           
            print("🗑️ Removendo arquivos temporários (fatias)...")
            for arquivo_velho in arquivos_json:
                try:
                    os.remove(arquivo_velho)
                    print(f"   Excluído: {os.path.basename(arquivo_velho)}")
                except Exception as e:
                    print(f"   Erro ao excluir {arquivo_velho}: {e}")
        
            with open(caminho_final, 'w', encoding='utf-8') as f_out:
                json.dump(dados_consolidados, f_out, indent=4, ensure_ascii=False)
            
            print("✨ Limpeza concluída com sucesso!")
        else:
            print("⚠️ Erro crítico: O arquivo final parece estar vazio. Abortando exclusão.")

    except Exception as e:
        print(f"❌ Erro ao salvar arquivo consolidado: {e}")
        


def consolidar_parquet(fonte, cidade, PASTA_DADOS, bairro = None):
    now = time.strftime("%Y-%m")
    
    if bairro is not None:
        padrao_busca = str(PASTA_DADOS / f'{cidade}_{bairro}_{fonte}_*.parquet')
    else:
        padrao_busca = str(PASTA_DADOS / f'{cidade}_{fonte}_*.parquet')
    
    arquivos_parquet = glob.glob(padrao_busca)
    
    if not arquivos_parquet:
        print(f"⚠️ Nenhum arquivo encontrado para {fonte} em {cidade}.")
        return

    lista_dfs = []
    total_arquivos = len(arquivos_parquet)

    print(f"Iniciando a união de {total_arquivos} arquivos Parquet...")

    for i, caminho in enumerate(arquivos_parquet, 1):
        nome_base = os.path.basename(caminho)
        try:
            # Lendo o parquet diretamente com pandas
            df_temp = pd.read_parquet(caminho)

                
            lista_dfs.append(df_temp)
            print(f"[{i}/{total_arquivos}] Adicionado: {nome_base} ({len(df_temp)} itens)")
            
        except Exception as e:
            print(f"❌ Erro ao ler {nome_base}: {e}")

    if not lista_dfs:
        print("⚠️ Nenhum dado válido encontrado nos arquivos.")
        return

    try:
        # 2. Une todos os DataFrames em um só
        df_consolidado = pd.concat(lista_dfs, ignore_index=True)

        if bairro is not None:
            nome_final = f'{cidade}_{bairro}_{fonte}_{now}.parquet'
        else:
            nome_final = f'{cidade}_{fonte}_{now}.parquet'
            
        caminho_final = PASTA_DADOS / nome_final

        # Salva usando compressão snappy (muito mais leve)
        df_consolidado.to_parquet(caminho_final, index=False, compression='snappy')
        
        # --- VALIDAÇÃO DE SEGURANÇA ---
        tamanho_final = os.path.getsize(caminho_final)
        
        if tamanho_final > 0:
            print(f"✅ Consolidação concluída: {len(df_consolidado)} registros.")
            print(f"📦 Arquivo gerado: {nome_final} ({tamanho_final / 1024 / 1024:.2f} MB)")
            
            # 4. Deleta os arquivos temporários (fatias)
            print("🗑️ Removendo arquivos temporários (fatias)...")
            for arquivo_velho in arquivos_parquet:
                # Evita deletar o próprio arquivo final caso ele tenha entrado no glob
                if os.path.abspath(arquivo_velho) != os.path.abspath(caminho_final):
                    try:
                        os.remove(arquivo_velho)
                        print(f"   Excluído: {os.path.basename(arquivo_velho)}")
                    except Exception as e:
                        print(f"   Erro ao excluir {arquivo_velho}: {e}")
            
            print("✨ Limpeza concluída com sucesso!")
            
            df_consolidado.to_parquet(caminho_final, index=False, compression='snappy')
        
        else:
            print("⚠️ Erro crítico: O arquivo final parece estar vazio. Abortando exclusão.")
        

    except Exception as e:
        print(f"❌ Erro na união ou salvamento: {e}")


In [10]:
def normalizar_bairros(bairro, mapeamento):
    if not isinstance(bairro, str):
        return bairro
        
    bairro_low = bairro.lower()
    
    for nome_correto, variacoes in mapeamento.items():
        # Verifica se qualquer uma das variações está contida no nome original
        if any(v in bairro_low for v in variacoes):
            return nome_correto
            
    return bairro 

async def limpando_dados(name_arquivo_zap : str, 
         name_arquivo_vivareal: str, 
         name_arquivo_chave_mao: str,
         name_arquivo_olx: str,
         name_arquivo_saida: str,
         pasta_dados : Path, 
         batch: int = 1,
         tipo_async: bool = False,
         cidade_localizacao: str = 'Joinville', 
         cidade_limpeza: str = 'joinville',
         estado_limpeza: str = 'sc', 
         estado_localizacao: str = 'SC',
         pais: str = 'Brasil', 
         MAPA_BAIRROS: dict = None,):
    
    logger.info(f"Iniciando limpeza de dados de imóveis de {cidade_limpeza}...")
    
    logger.info(f"Pasta de dados: {pasta_dados}")

    #pasta_dados.mkdir(parents=True, exist_ok=True)

    df_zap, arquivo_zap      = carregar_parquet(pasta_dados, name_arquivo_zap)
    
    df_vivareal, arquivo_vivareal = carregar_parquet(pasta_dados,name_arquivo_vivareal)
    
    df_chave_mao, arquivo_chave_mao = carregar_parquet(pasta_dados,name_arquivo_chave_mao)
    
    df_olx, arquivo_olx = carregar_json(pasta_dados,name_arquivo_olx)

    if not df_zap.empty:
        df_zap['fonte'] = 'zap_imoveis'
    if not df_vivareal.empty:
        df_vivareal['fonte'] = 'viva_real'

    if not df_chave_mao.empty:
        df_chave_mao['fonte'] = 'chave_mao'
    
    if not df_olx.empty:
        df_olx['fonte'] = 'olx'

    if df_zap.empty and df_vivareal.empty and df_chave_mao.empty and df_olx.empty:
        logger.error("Nenhum dado encontrado em nenhuma das fontes — abortando.")
        return
    
    #df_zap_endereco_limpo = df_zap['endereco'].apply(limpa_endereco_apply_zap)
    #df_vivareal_endereco_limpo = df_vivareal['endereco'].apply(limpa_endereco_apply_zap)
    #df_chave_mao_endereco_limpo = df_chave_mao['endereco'].apply(limpa_endereco_apply_chave_mao)
    #df_olx_endereco_limpo = df_olx['endereco'].apply(limpa_endereco_apply_olx)
    
    df_zap_endereco_limpo = df_zap['endereco'].apply(lambda x: limpa_endereco_apply_zap(x, cidade_limpeza, estado_limpeza))
    df_vivareal_endereco_limpo = df_vivareal['endereco'].apply(lambda x: limpa_endereco_apply_zap(x, cidade_limpeza, estado_limpeza))
    df_chave_mao_endereco_limpo = df_chave_mao['endereco'].apply(lambda x: limpa_endereco_apply_chave_mao(x, cidade_limpeza, estado_limpeza))
    df_olx_endereco_limpo = df_olx['endereco'].apply(lambda x: limpa_endereco_apply_olx(x, cidade_limpeza, estado_limpeza))
    
    df_zap_endereco =  pd.concat([df_zap, df_zap_endereco_limpo], axis=1)
    df_vivareal_endereco =  pd.concat([df_vivareal, df_vivareal_endereco_limpo], axis=1)
    df_chave_mao_endereco =  pd.concat([df_chave_mao, df_chave_mao_endereco_limpo], axis=1)
    df_olx_endereco =  pd.concat([df_olx, df_olx_endereco_limpo], axis=1)

    df = pd.concat([df_zap_endereco if not df_zap_endereco.empty else pd.DataFrame(), 
                    df_vivareal_endereco if not df_vivareal_endereco.empty else pd.DataFrame(),
                    df_chave_mao_endereco if not df_chave_mao_endereco.empty else pd.DataFrame(),
                    df_olx_endereco if not df_olx_endereco.empty else pd.DataFrame()], 
                   axis=0, ignore_index=True)
    
    logger.info(f"Total de registros carregados: {len(df)} (zap: {len(df_zap)} | vivareal: {len(df_vivareal)} | chave_mao: {len(df_chave_mao)} | olx: {len(df_olx)})")

    # Limpeza
    df_limpo = await limpando_dados_cidades(df, 
                                        batch = batch, 
                                        cidade_limpeza= cidade_limpeza, 
                                        cidade_localizacao= cidade_localizacao,
                                        tipo_async=tipo_async,
                                        estado_limpeza= estado_limpeza,
                                        estado_localizacao= estado_localizacao, 
                                        pais= pais
                                        )

    # Remove duplicatas
    colunas_dedup = ['valor_imovel', 'rua', 'bairro', 'metragem', 'quartos', 'preco_por_m2', 'banheiros', 'lat', 'lng']
    
    colunas_dedup = [c for c in colunas_dedup if c in df_limpo.columns]  

    antes = len(df_limpo)
    
    df_limpo = df_limpo.drop_duplicates(subset=colunas_dedup, keep='first').reset_index(drop=True)
    
    logger.info(f"Duplicatas removidas: {antes - len(df_limpo)} | Registros finais: {len(df_limpo)}")
    
    if MAPA_BAIRROS:
        df_limpo['bairro'] = df_limpo['bairro'].apply(normalizar_bairros, args=(MAPA_BAIRROS,))
    
    logger.info(f"Coluna 'bairro' corrigida...")
    
    #df_limpo = df_limpo.groupby('bairro').filter(lambda x: len(x) > 1)
    
    return df_limpo




In [11]:
import time
from datetime import datetime
import json
from pathlib import Path

import asyncio
import sys
from pathlib import Path
import time
import pandas as pd

bairro = 'barra-tijuca'

cidade = 'rio_janeiro'

estado = 'rj'

cidade_limpeza = 'rio de janeiro'

estado_limpeza = 'rj'

cidade_localizacao = 'Rio de Janeiro'

estado_localizacao = 'RJ'

pais = 'Brasil'

BASE_DIR = Path.cwd().parent

PASTA_DADOS= BASE_DIR / 'dados' / cidade / bairro

#pd_data = pd.read_parquet(pasta_dados / f'sao_paulo_{bairro}_imoveis_limpo_2026-05.parquet')



In [12]:
"""PASTA_DADOS.mkdir(parents=True, exist_ok=True)

logger.info(f"Diretório de dados: {PASTA_DADOS}")

consolidar_parquet('vivareal', cidade, PASTA_DADOS, bairro)

logger.info(f"Dados consolidados para Viva Real.")

consolidar_parquet('chave_mao', cidade, PASTA_DADOS, bairro)

logger.info(f"Dados consolidados para Chave na Mão.")

consolidar_parquet('zap', cidade, PASTA_DADOS, bairro)

logger.info(f"Dados consolidados para Zap.")

consolidar_jsons('olx', cidade, PASTA_DADOS, bairro)

logger.info(f"Dados consolidados para Olx.")

logger.info(f"Arquivos consolidados em: {PASTA_DADOS}")

"""

'PASTA_DADOS.mkdir(parents=True, exist_ok=True)\n\nlogger.info(f"Diretório de dados: {PASTA_DADOS}")\n\nconsolidar_parquet(\'vivareal\', cidade, PASTA_DADOS, bairro)\n\nlogger.info(f"Dados consolidados para Viva Real.")\n\nconsolidar_parquet(\'chave_mao\', cidade, PASTA_DADOS, bairro)\n\nlogger.info(f"Dados consolidados para Chave na Mão.")\n\nconsolidar_parquet(\'zap\', cidade, PASTA_DADOS, bairro)\n\nlogger.info(f"Dados consolidados para Zap.")\n\nconsolidar_jsons(\'olx\', cidade, PASTA_DADOS, bairro)\n\nlogger.info(f"Dados consolidados para Olx.")\n\nlogger.info(f"Arquivos consolidados em: {PASTA_DADOS}")\n\n'

In [13]:
pd_data = await limpando_dados(name_arquivo_zap = f'{cidade}_{bairro}_zap_*.parquet', 
               name_arquivo_vivareal = f'{cidade}_{bairro}_vivareal_*.parquet', 
               name_arquivo_chave_mao = f'{cidade}_{bairro}_chave_mao_*.parquet',
               name_arquivo_olx = f'{cidade}_{bairro}_olx_*.json',
               name_arquivo_saida = f'{cidade}_{bairro}_imoveis_limpo', 
               pasta_dados = PASTA_DADOS, 
               tipo_async = True,
               batch = 100, 
               cidade_limpeza=cidade_limpeza,
               cidade_localizacao=cidade_localizacao,
               estado_limpeza=estado_limpeza,
               estado_localizacao=estado_localizacao, )

2026-05-18 16:18:17,912 - INFO - Iniciando limpeza de dados de imóveis de rio de janeiro...
2026-05-18 16:18:17,913 - INFO - Pasta de dados: c:\Users\jefer\Documents\Ciencia-de-dados\Preco-Imoveis\dados\rio_janeiro\barra-tijuca
2026-05-18 16:18:17,913 - INFO - Arquivo Parquet encontrado: rio_janeiro_barra-tijuca_zap_2026-05.parquet
2026-05-18 16:18:17,974 - INFO - Arquivo Parquet encontrado: rio_janeiro_barra-tijuca_vivareal_2026-05.parquet
2026-05-18 16:18:18,015 - INFO - Arquivo Parquet encontrado: rio_janeiro_barra-tijuca_chave_mao_2026-05.parquet
2026-05-18 16:18:18,382 - INFO - Arquivo encontrado: rio_janeiro_barra-tijuca_olx_2026-05.json
2026-05-18 16:18:26,266 - INFO - Total de registros carregados: 48561 (zap: 2381 | vivareal: 2921 | chave_mao: 14041 | olx: 29218)
2026-05-18 16:18:26,268 - INFO - Iniciando o processo de limpeza de dados de imóveis...
2026-05-18 16:18:26,419 - INFO - Removendo linhas com muitos valores faltantes. Registros restantes: (44054, 22)
2026-05-18 16:18

In [ ]:
pd_data['bairro'].value_counts()

bairro
barra da tijuca             31159
barra olimpica               2962
recreio dos bandeirantes      787
itanhanga                     199
jacarepagua                    47
joa                            23
freguesia (jacarepagua)        22
vargem pequena                 17
anil                           11
centro                          9
camorim                         8
pechincha                       7
taquara                         7
curicica                        4
tijuca                          2
jardim oceanico                 1
tanque                          1
sao conrado                     1
s/b                             1
Name: count, dtype: int64

: 

In [27]:
def normalizar_bairros(bairro, mapeamento):
    if not isinstance(bairro, str):
        return bairro
        
    bairro_low = bairro.lower()
    
    for nome_correto, variacoes in mapeamento.items():
        # Verifica se qualquer uma das variações está contida no nome original
        if any(v in bairro_low for v in variacoes):
            return nome_correto
            
    return bairro 

mapa_bairros = {'jardins' :['jardins', 'jardim']}

pd_bairro = pd_data.copy()

pd_bairro['bairro'] = pd_bairro['bairro'].apply(normalizar_bairros, args=(mapa_bairros,))

In [36]:
pd_bairro = pd_bairro[pd_bairro['bairro'].isin(['jardins'])]

In [ ]:
#pd_data_limpo = pd_data[pd_data['bairro'].isin(['moema'])]

In [ ]:
#pd_bairro.to_parquet(PASTA_DADOS / f'sao_paulo_{bairro}_imoveis_limpo_2026-05.parquet')

In [ ]:
PASTA_DADOS = BASE_DIR / 'dados' / "curitiba"
pd_curitiba = pd.read_parquet(PASTA_DADOS / f'curitiba_imoveis_limpo_2026-05.parquet')

In [38]:
pd_completo = pd.read_parquet(PASTA_DADOS / f'sao_paulo_{bairro}_imoveis_limpo_2026-05.parquet')

In [39]:
pd_completo

,url,titulo,metragem,banheiros,vagas,quartos,valor_imovel,condominio,endereco,iptu,...,dias_publicacao,tipo_imovel,lat,lng,preco_por_m2,faixa,p25_bairro,p50_bairro,p75_bairro,desvio_mediana
0,https://www.zapimoveis.com.br/imovel/venda-fla...,"Flat com 2 Quartos à venda, 110m² - Jardins",110.0,2,3,2,1390000.0,5800.0,"Rua Batataes, 159 - Jardins, São Paulo - SP",1100.0,...,172.0,apartamento,-23.570938,-46.656678,12636.363636,barato,15000.000000,20169.540230,29444.444444,-0.37
1,https://www.zapimoveis.com.br/imovel/venda-con...,"Conjunto Comercial / Sala à venda, 109m² - Jar...",109.0,3,2,0,1385000.0,5149.0,"Alameda Jaú - Jardins, São Paulo - SP",2400.0,...,38.0,comercial,-23.567325,-46.653397,12706.422018,barato,12706.422018,15000.000000,26190.476190,-0.15
2,https://www.zapimoveis.com.br/imovel/venda-fla...,"Flat com 2 Quartos à venda, 110m² - Jardins",110.0,2,1,2,950000.0,3061.0,"Alameda Jaú, 1474 - Jardins, São Paulo - SP",435.0,...,31.0,apartamento,-23.567325,-46.653397,8636.363636,barato,15000.000000,20169.540230,29444.444444,-0.57
3,https://www.zapimoveis.com.br/imovel/venda-fla...,"Flat com 2 Quartos à venda, 104m² - Jardins",104.0,2,2,2,2250000.0,2100.0,"Alameda Itu, 78 - Jardins, São Paulo - SP",0.0,...,16.0,apartamento,-23.567188,-46.655413,21634.615385,medio_alto,15000.000000,20169.540230,29444.444444,0.07
4,https://www.zapimoveis.com.br/imovel/venda-apa...,"Apartamento com 4 Quartos à venda, 110m² - Jar...",110.0,3,1,4,1280000.0,1279.0,"Alameda Ribeirão Preto, 267 - Jardins, São Pau...",253.0,...,604.0,apartamento,-23.573808,-46.667831,11636.363636,barato,15000.000000,20169.540230,29444.444444,-0.42
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19072,https://sp.olx.com.br/sao-paulo-e-regiao/imove...,"Cobertura Duplex á venda, com 985 m, 4 suítes ...",985.0,0,0,5,26000000.0,15203.0,"Jardim Paulista, São Paulo, SP, 01435010",6201.0,...,NaN,apartamento,-30.806798,-53.898964,26395.939086,alto_padrao,12692.307692,15967.741935,21428.571429,0.65
19073,https://sp.olx.com.br/sao-paulo-e-regiao/imove...,Prédio inteiro na Brigadeiro Luis Antonio!,1000.0,0,0,0,18800000.0,0.0,"Jardim Paulista, São Paulo, SP, 01402002",74723.0,...,NaN,predio_inteiro,-23.581250,-46.665575,18800.000000,medio_baixo,13636.363636,24285.714286,36250.000000,-0.23
19074,https://sp.olx.com.br/sao-paulo-e-regiao/imove...,EDIFÍCIO LORENA - INVESTIMENTO COM ALTO POTENC...,1000.0,0,0,0,18800000.0,0.0,"Jardim Paulista, São Paulo, SP, 01401001",82196.0,...,NaN,apartamento,-23.575664,-46.657329,18800.000000,medio_alto,12692.307692,15967.741935,21428.571429,0.18
19075,https://sp.olx.com.br/sao-paulo-e-regiao/imove...,"Prédio comercial no Jardim Paulista , na Av. B...",1000.0,0,0,0,18800000.0,0.0,"Jardim Paulistano, São Paulo, SP, 01451000",7473.0,...,NaN,comercial,-23.576107,-46.687885,18800.000000,barato,18800.000000,18800.000000,22650.000000,0.00
